# Credit Risk Model Studio: training notebook

Same code as the `ML.TRAIN_MODELS` procedure in `sql/setup.sql`, laid out for a data scientist in a **Snowflake Notebook on Container Runtime** (CPU is enough):

1. Register the `LOAN` entity and two Snowflake-managed feature views (Dynamic Tables) in the Feature Store
2. Build a point-in-time training set from `ML.TRAINING_SPINE`
3. Train champion (V1) and challenger (V2) XGBoost models with a small sweep, and compare them out of time
4. Log both to the Model Registry with metrics, explainability, and `CHAMPION` / `CHALLENGER` aliases

Synthetic data and an illustrative model only. Run `sql/setup.sql` first. It creates the data, the role, and the `EXPERIMENT_RUNS` / `MODEL_EVAL` tables this code writes to. The pack's tested live path is the stored procedure; this notebook carries the same code for teams that prefer to iterate interactively.

In [ ]:
# In a Snowflake Notebook, use the active session and the pack's role and warehouse.
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql('USE ROLE CREDIT_RISK_STUDIO_ROLE').collect()
session.sql('USE WAREHOUSE CREDIT_RISK_STUDIO_WH').collect()
session.sql('USE SCHEMA CREDIT_RISK_STUDIO.ML').collect()

## Feature Store, training, and registry
This cell defines the helpers and `run(session)` exactly as the stored procedure does.

In [ ]:
import json, numpy as np, pandas as pd, xgboost as xgb
from sklearn.metrics import roc_auc_score, roc_curve
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode
from snowflake.ml.registry import Registry
from snowflake.ml.model import task

DB, WH = 'CREDIT_RISK_STUDIO', 'CREDIT_RISK_STUDIO_WH'
CHAMP = ['FICO', 'UTILIZATION', 'INQUIRIES_6M', 'TRADELINES', 'DTI', 'APR', 'LOAN_AMOUNT', 'TERM_MONTHS',
         'IS_AUTO', 'MONTHS_ON_BOOK', 'MAX_DPD_6M', 'PAY_RATIO_3M']
CHALL = CHAMP + ['MONTHS_ON_BUREAU']

def metrics(y, p):
    fpr, tpr, _ = roc_curve(y, p)
    auc = roc_auc_score(y, p)
    return {'auc': round(float(auc), 4), 'ks': round(float(np.max(tpr - fpr)), 4), 'gini': round(float(2 * auc - 1), 4),
            'n': int(len(y)), 'defaultRate': round(float(np.mean(y)), 4)}

def evaluate(y, p, seg):
    out = metrics(y, p)
    order = np.argsort(p); deciles = np.array_split(order, 10)
    out['calibration'] = [{'decile': i + 1, 'predicted': round(float(p[d].mean()), 4), 'observed': round(float(y[d].mean()), 4)} for i, d in enumerate(deciles)]
    out['fairness'] = []
    for s in ['THICK', 'THIN', 'NEW_TO_CREDIT']:
        m = seg == s
        out['fairness'].append({'segment': s, 'n': int(m.sum()), 'observed': round(float(y[m].mean()), 4),
                                'predicted': round(float(p[m].mean()), 4), 'auc': round(float(roc_auc_score(y[m], p[m])), 4)})
    return out

def run(session):
    fs = FeatureStore(session=session, database=DB, name='FEATURES', default_warehouse=WH,
                      creation_mode=CreationMode.CREATE_IF_NOT_EXIST)
    loan = Entity(name='LOAN', join_keys=['LOAN_ID'], desc='A consumer loan (personal or auto)')
    fs.register_entity(loan)

    attrs_df = session.sql("""
        SELECT LOAN_ID, DTI::FLOAT AS DTI, APR::FLOAT AS APR, LOAN_AMOUNT::FLOAT AS LOAN_AMOUNT,
               TERM_MONTHS::FLOAT AS TERM_MONTHS, IFF(PRODUCT = 'AUTO', 1, 0)::FLOAT AS IS_AUTO
        FROM CREDIT_RISK_STUDIO.RAW.LOANS""")
    behavior_df = session.sql("""
        SELECT p.LOAN_ID, p.STATEMENT_MONTH AS OBS_DATE,
               b.FICO::FLOAT AS FICO, b.REVOLVING_UTILIZATION::FLOAT AS UTILIZATION, b.INQUIRIES_6M::FLOAT AS INQUIRIES_6M,
               b.OPEN_TRADELINES::FLOAT AS TRADELINES, p.MONTHS_ON_BOOK::FLOAT AS MONTHS_ON_BOOK,
               b.MONTHS_ON_BUREAU::FLOAT AS MONTHS_ON_BUREAU,
               MAX(p.DAYS_PAST_DUE) OVER (PARTITION BY p.LOAN_ID ORDER BY p.STATEMENT_MONTH ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)::FLOAT AS MAX_DPD_6M,
               ROUND(SUM(p.PAID_AMOUNT) OVER (PARTITION BY p.LOAN_ID ORDER BY p.STATEMENT_MONTH ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
                   / SUM(p.SCHEDULED_AMOUNT) OVER (PARTITION BY p.LOAN_ID ORDER BY p.STATEMENT_MONTH ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 4)::FLOAT AS PAY_RATIO_3M
        FROM CREDIT_RISK_STUDIO.RAW.PAYMENTS_MONTHLY p
        JOIN CREDIT_RISK_STUDIO.RAW.BUREAU_SNAPSHOTS_MONTHLY b ON b.LOAN_ID = p.LOAN_ID AND b.SNAPSHOT_MONTH = p.STATEMENT_MONTH""")
    fv_attr = fs.register_feature_view(
        FeatureView(name='LOAN_ATTRIBUTES', entities=[loan], feature_df=attrs_df, refresh_freq='1 day',
                    desc='Static loan terms at origination'), version='1', block=True, overwrite=True)
    fv_beh = fs.register_feature_view(
        FeatureView(name='LOAN_BEHAVIOR', entities=[loan], feature_df=behavior_df, timestamp_col='OBS_DATE', refresh_freq='1 day',
                    desc='Monthly bureau and payment behavior with 3- and 6-month windows'), version='1', block=True, overwrite=True)

    spine = session.table('CREDIT_RISK_STUDIO.ML.TRAINING_SPINE')
    ts = fs.generate_training_set(spine_df=spine, features=[fv_attr, fv_beh], spine_timestamp_col='OBS_DATE',
                                  spine_label_cols=['DEFAULTED_12M']).to_pandas()
    ts = ts.sort_values('LOAN_ID').reset_index(drop=True)
    oot = ts[ts.VINTAGE_INDEX >= 27]
    session.sql('TRUNCATE TABLE CREDIT_RISK_STUDIO.ML.EXPERIMENT_RUNS').collect()
    session.sql('TRUNCATE TABLE CREDIT_RISK_STUDIO.ML.MODEL_EVAL').collect()

    reg = Registry(session=session, database_name=DB, schema_name='ML')
    # Re-runs start clean: the monitor depends on the model version, so drop it first.
    session.sql('DROP MODEL MONITOR IF EXISTS CREDIT_RISK_STUDIO.ML.CREDIT_DEFAULT_V1_MONITOR').collect()
    try:
        reg.delete_model('CREDIT_DEFAULT_MODEL')
    except Exception:
        pass
    results = {}
    specs = [('V1', 'CHAMPION', CHAMP, ts[ts.VINTAGE_INDEX <= 20], '2023-01 to 2024-09 vintages'),
             ('V2', 'CHALLENGER', CHALL, ts[ts.VINTAGE_INDEX <= 26], '2023-01 to 2025-03 vintages')]
    for version, alias, cols, train, window in specs:
        # Small hyperparameter sweep; every run is recorded, the best OOT run is logged to the registry.
        best = None
        for i, (depth, lr, n) in enumerate([(3, 0.08, 250), (4, 0.06, 300), (5, 0.05, 300)]):
            clf = xgb.XGBClassifier(max_depth=depth, learning_rate=lr, n_estimators=n, subsample=0.8, colsample_bytree=0.8,
                                    min_child_weight=20, random_state=7, n_jobs=1, eval_metric='auc')
            clf.fit(train[cols].astype('float64'), train.DEFAULTED_12M)
            m = metrics(oot.DEFAULTED_12M.values, clf.predict_proba(oot[cols].astype('float64'))[:, 1])
            run_id = f'{version}-run{i + 1}'
            session.sql('INSERT INTO CREDIT_RISK_STUDIO.ML.EXPERIMENT_RUNS SELECT ?, ?, PARSE_JSON(?), ?, ?, ?, FALSE, CURRENT_TIMESTAMP()::TIMESTAMP_NTZ',
                        params=[run_id, version, json.dumps({'max_depth': depth, 'learning_rate': lr, 'n_estimators': n}), m['auc'], m['ks'], m['gini']]).collect()
            if best is None or m['auc'] > best[1]['auc']:
                best = (clf, m, run_id)
        clf, _, run_id = best
        session.sql('UPDATE CREDIT_RISK_STUDIO.ML.EXPERIMENT_RUNS SET LOGGED = TRUE WHERE RUN_ID = ?', params=[run_id]).collect()
        X = train[cols].astype('float64')
        p_tr = clf.predict_proba(X)[:, 1]
        p_oot = clf.predict_proba(oot[cols].astype('float64'))[:, 1]
        ev = evaluate(oot.DEFAULTED_12M.values, p_oot, oot.FILE_TYPE.values)
        imp = clf.get_booster().get_score(importance_type='gain')
        tot = sum(imp.values())
        importance = sorted([{'feature': k, 'importance': round(v / tot, 4)} for k, v in imp.items()], key=lambda r: -r['importance'])
        mv = reg.log_model(clf, model_name='CREDIT_DEFAULT_MODEL', version_name=version,
                           sample_input_data=X.sample(1000, random_state=7),  # background data for explain (Preview)
                           task=task.Task.TABULAR_BINARY_CLASSIFICATION,
                           conda_dependencies=['xgboost'],
                           target_platforms=['WAREHOUSE'],              # explain runs only in a warehouse
                           options={'enable_explainability': True},     # set explicitly; can't be added after logging
                           comment=f'{alias.title()}: XGBoost trained on {window}',
                           metrics={'oot_auc': ev['auc'], 'oot_ks': ev['ks'], 'oot_gini': ev['gini'], 'train_rows': int(len(train))})
        session.sql(f'ALTER MODEL CREDIT_RISK_STUDIO.ML.CREDIT_DEFAULT_MODEL VERSION {version} SET ALIAS = {alias}').collect()
        payload = {'version': version, 'alias': alias, 'algorithm': 'XGBoost classifier', 'trainingWindow': window,
                   'features': cols, 'trainRows': int(len(train)), 'bestRun': run_id,
                   'train': metrics(train.DEFAULTED_12M.values, p_tr), 'oot': ev, 'importance': importance,
                   'featureViews': ['FEATURES.LOAN_ATTRIBUTES$1', 'FEATURES.LOAN_BEHAVIOR$1']}
        session.sql('INSERT INTO CREDIT_RISK_STUDIO.ML.MODEL_EVAL SELECT ?, PARSE_JSON(?), CURRENT_TIMESTAMP()::TIMESTAMP_NTZ',
                    params=[version, json.dumps(payload)]).collect()
        results[version] = {'alias': alias, 'oot_auc': ev['auc'], 'run': run_id}
    session.sql('ALTER MODEL CREDIT_RISK_STUDIO.ML.CREDIT_DEFAULT_MODEL SET DEFAULT_VERSION = V1').collect()
    return results

## Run it
Trains both versions, records every sweep run in `ML.EXPERIMENT_RUNS`, and logs the best run of each version to the registry.

In [ ]:
results = run(session)
results

## Inspect the registry
Aliases decide which version the scoring task uses; promotion moves `CHAMPION` without code changes.

In [ ]:
from snowflake.ml.registry import Registry
reg = Registry(session=session, database_name='CREDIT_RISK_STUDIO', schema_name='ML')
m = reg.get_model('CREDIT_DEFAULT_MODEL')
m.show_versions()[['name', 'aliases', 'comment']]

In [ ]:
# Shapley reason codes (explain is a Preview feature) for a few recent accounts
mv = m.version('CHALLENGER')
cols = [f.name for f in mv.show_functions()[0]['signature'].inputs]
x = session.sql('''SELECT f.*, a.DTI, a.APR, a.LOAN_AMOUNT, a.TERM_MONTHS, a.IS_AUTO
  FROM CREDIT_RISK_STUDIO.FEATURES."LOAN_BEHAVIOR$1" f JOIN CREDIT_RISK_STUDIO.FEATURES."LOAN_ATTRIBUTES$1" a USING (LOAN_ID)
  WHERE f.OBS_DATE = '2026-09-01' LIMIT 5''').to_pandas()[cols]
mv.run(x, function_name='explain')